In [1]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

# 禁用所有警告
warnings.filterwarnings("ignore")

In [2]:
way = 'max'  # Med 可以设置为 'max'、'median' 或 'min'

def median_chart_process(ts):
    # 将所有列值全部为 0 的列替换为 NaN
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  

    # 处理列数据
    processed_values = {}
    for col in ts.columns:
        if col.startswith('CHART_Cate_'):  
            # 对于 'CHART_Cate_' 开头的列，取最大值
            processed_values[col] = ts[col].max()
        elif col.startswith('CHART_Str_'):  
            # 对于 'CHART_Str_' 开头的列，取除 0 外出现次数最多的值
            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:
            # 根据 'way' 的值来决定使用的统计量
            if way == 'median':
                processed_values[col] = ts[col].median()
            elif way == 'min':
                processed_values[col] = ts[col].min()
            else:  # 默认使用 'max'
                processed_values[col] = ts[col].max()
    
    # 将结果转为 DataFrame
    result = pd.DataFrame([processed_values])
    
    return result

In [3]:
# CHARTS
# 24 hr
all_ts = pd.DataFrame()

# 遍历每个 ICUSTAY_ID
for i in tqdm(os.listdir('D:/2024BI_Journal/MIMIC_IV/Final_Chart_2/')):
    # 读取 dynamic.csv 文件
    file_path = f'D:/2024BI_Journal/MIMIC_IV/Final_Chart_2/{i}'
    ts = pd.read_csv(file_path)
    
    if ts.shape[0]>0:
        ts = ts.iloc[:96,:]

        # 调用 ts_process 函数进行数据处理
        ts = median_chart_process(ts)

        ts['ICUSTAY_ID'] = i.split('.')[0]

        # 合并处理后的数据
        all_ts = pd.concat([all_ts, ts], ignore_index=True)
    
all_ts = all_ts.dropna(axis=1, how='all')

100%|████████████████████████████████████████████████████████████████████████████| 74827/74827 [54:50<00:00, 22.74it/s]


In [4]:
all_ts.shape

(74827, 84)

In [5]:
all_ts.head(2)

,CHART_220210,CHART_220045,CHART_220179,CHART_220180,CHART_220181,CHART_220277,CHART_220339,CHART_224690,CHART_224697,CHART_223834,...,CHART_Cate_225126,CHART_224144,CHART_223762,CHART_229355,CHART_Cate_225108,CHART_227428,CHART_226755,CHART_Str_226756,CHART_Str_226757,CHART_Str_226758
0,15.0,107.5,113.0,77.0,84.0,96.0,5.0,14.0,6.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19.0,83.0,130.0,55.0,78.0,99.0,8.0,22.0,11.6,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
all_ts.to_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/A24_Chart_median_max_nan.csv',index=False)